# 08 - Docker and real HTTP

Use the saved bundle from 04. Build a local image, check API responses, and measure warm requests at concurrency 10. This step downloads a Python image and packages, but does not change the notebook environment or create Azure resources.


## 1. Check inputs

Use the same WSL kernel as 07. Docker must be running. The bundle and golden profiles must match.


In [1]:
from pathlib import Path
import json
import platform
import shutil
import subprocess
import sys

assert platform.system() == "Linux", "Use the WSL kernel."
assert Path(sys.prefix).name == "mlop-pj", "Select the project environment."
ROOT = next(p for p in [Path.cwd(), *Path.cwd().parents]
            if p.name == "MLOP-PJ" and (p / "README.md").is_file())
assert shutil.which("docker"), "Docker CLI is missing."
subprocess.run(["docker", "version", "--format", "{{.Server.Version}}"], check=True)
manifest = json.loads((ROOT / "reports/bundle-manifest.json").read_text())
fixture = json.loads((ROOT / "reports/golden-profiles.json").read_text())
assert fixture["model_version"] == manifest["model_version"]
assert fixture["artifact_sha256"] == manifest["artifact_sha256"]
print("Ready to build model", manifest["model_version"])


29.4.0
Ready to build model abfd88e893ca


## 2. Build and test

Run this cell once. The first build may take several minutes. It pins the downloaded base image by digest and uses the installed service versions. The API runs on a random localhost port with 1 CPU and 512 MB.

Checks: health, readiness, golden predictions, bad input, and batch 100. Then send 10 warm-up requests and 200 measured requests with up to 10 clients. The test container stops at the end; its image and logs stay local. Stop if this cell fails. Read `reports/docker-build.txt` for build errors or `reports/docker-api.log` for startup errors.


In [2]:
print("Building and checking Docker. Wait for this cell to finish.", flush=True)
result = subprocess.run([sys.executable, "scripts/docker_check.py",
                         "--requests", "200", "--concurrency", "10"],
                        cwd=ROOT, capture_output=True, text=True)
print(result.stdout)
if result.stderr:
    print(result.stderr)
assert result.returncode == 0, "Docker or HTTP checks failed. Read the output and logs."


Building and checking Docker. Wait for this cell to finish.
Pulling python:3.13-slim and building the API image. This may take a few minutes.
HTTP smoke passed. Measuring 200 requests at concurrency 10 by default.
{
  "scope": "local real HTTP; not Azure performance",
  "model_version": "abfd88e893ca",
  "requests": 200,
  "concurrency": 10,
  "batch_size": 1,
  "warmup_requests": 10,
  "errors": 0,
  "error_rate": 0.0,
  "duration_seconds": 8.425022800001898,
  "successful_requests_per_second": 23.73880816084616,
  "mean_ms": 415.497270260239,
  "p95_ms": 567.9912881045311,
  "latency_scope": "successful requests; client HTTP time including response parsing",
  "workload": "cycle five saved golden profiles; closed-loop clients",
  "target_p95_ms": 300,
  "target_met": false,
  "image_id": "sha256:136feea52ca376b6bc36ab8545f8c1293b4572309c82815fcbe7afba7b7dfb04",
  "created_at": "2026-10-11T12:13:08.481747+00:00"
}
The test container stopped. The local image and reports are kept.



## 3. Read the result

Expect smoke `passed: true`, status codes `[200, 200, 200, 422, 200]`, and load `errors: 0`. `target_met` checks warm p95 below 300 ms with no errors. If false, keep the result for review. This small local workload cycles five golden profiles; it does not prove Azure latency or real user performance. Startup-to-ready time includes Docker launch and polling, so it is not an Azure cold-start measurement.


In [ ]:
image = json.loads((ROOT / "reports/docker-image.json").read_text())
smoke = json.loads((ROOT / "reports/docker-smoke.json").read_text())
load = json.loads((ROOT / "reports/local-load.json").read_text())
assert smoke["passed"] and load["errors"] == 0
assert image["image_id"] == smoke["image_id"] == load["image_id"]
assert image["model_version"] == smoke["model_version"] == load["model_version"]
summary = {"model_version": image["model_version"], "image_id": image["image_id"],
           "smoke_passed": smoke["passed"], "startup_to_ready_ms": smoke["startup_to_ready_ms"],
           **{key: load[key] for key in ["requests", "concurrency", "errors", "mean_ms", "p95_ms", "target_met"]}}
print(json.dumps(summary, indent=2))


{
  "model_version": "abfd88e893ca",
  "image_id": "sha256:136feea52ca376b6bc36ab8545f8c1293b4572309c82815fcbe7afba7b7dfb04",
  "smoke_passed": true,
  "startup_to_ready_ms": 4911.551551005687,
  "requests": 200,
  "concurrency": 10,
  "errors": 0,
  "mean_ms": 415.497270260239,
  "p95_ms": 567.9912881045311,
  "target_met": false
}


## Output

Save this notebook. Evidence: `docker-image.json`, `docker-smoke.json`, `local-load.json`, build logs, and API logs in `reports/`. Next: review Azure region, quota, permissions, and cost before creating resources.
